# ===============================
# Mini-Agentic Car Manual Assistant
# ===============================
This is a small prototype of a car manual assistant that demonstrates how to build and use a retrieval-augmented generation (RAG) system using the FAISS (Facebook AI Similarity Search) library.

It accepts a user question (e.g. "How do I change the oil in my Mazda?") and returns an answer, which is constructed as follows:
1. It retrieves the top five chunks from the vector store that are most similar to the user question, ranked by similarity.
2. It combines these five chunks and summarises them to generate a concise answer.

The assistant consists of two agents and an orchestrator:
* Retriever Agent: takes the question, searches the vector store, and returns relevant chunks.
* Answer Agent: takes the retrieved chunks and the question, producing a user-friendly answer.
* Orchestrator: coordinates the Retriever and Answer agents.

#### History / context:
The assistant began as a straightforward Python prototype in Colab.
After experimenting with LangChain integration, it evolved into this quasi-agentic form, manually orchestrating retrieval and summarization without relying on LangChain.

#### Note:
* This prototype accurately portrays the importance of the quality of the data stored in a RAG system.
The Mazda car manual PDF used here is a simple one, so the quality of the assistant's answers depends directly on the content of the PDF.
However, this also demonstrates that the assistant faithfully reflects the PDF content and does not hallucinate information. In other words, "garbage in, garbage out." High-quality source data is essential for reliable RAG system performance.
* This prototype only extracts text from the PDF. Images, diagrams, and any callouts or text embedded within graphics are not processed. No OCR is applied to extract text from images, so all answers are based solely on the textual content of the manual.

In [1]:
# -------------------------------
# 1. Install dependencies
# -------------------------------
# Install the packages required for:
# - extracting text from PDFs (PyMuPDF)
# - creating embeddings from text (sentence-transformers)
# - storing and searching embeddings (faiss-cpu)
# - running the LLM on Groq (groq)
!pip install PyMuPDF sentence-transformers faiss-cpu groq
#!pip install PyMuPDF==1.22.5 sentence-transformers==2.2.2 faiss-cpu==1.7.3 groq==0.10.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 6.5 MB/s eta 0:00:00


In [16]:
# -------------------------------
# 2. Imports
# -------------------------------
# Load the modules/classes into the Colab environment,
import fitz           # PyMuPDF
from sentence_transformers import SentenceTransformer
import faiss          # faiss-cpu
import numpy as np
from groq import Groq
from google.colab import drive, userdata

In [13]:
# View the version number of some of the packages.

# PyMuPDF (fitz) – __version__ may not exist, fallback to __doc__
print("PyMuPDF version:", getattr(fitz, "__version__", fitz.__doc__.split()[0]))

# Sentence Transformers – class does not have __version__, use importlib.metadata
from importlib.metadata import version
print("Sentence Transformers version:", version("sentence-transformers"))

# FAISS – has __version__
print("FAISS version:", faiss.__version__)

# Groq – has __version__
print("Groq version:", groq.__version__)

PyMuPDF version: 1.27.1
Sentence Transformers version: 5.2.3
FAISS version: 1.13.2
Groq version: 1.0.0


In [17]:
# -------------------------------
# 3. Mount Google Drive & setup Groq
# -------------------------------
drive.mount('/content/drive/', force_remount=True)

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
groq_model = "llama-3.3-70b-versatile"

Mounted at /content/drive/


In [18]:
# -------------------------------
# 4. Load PDF & extract text
# -------------------------------
# Load the Mazda car manual PDF from Google Drive (obtained from https://www.carmans.net/2022-mazda-2/).
# This opens the PDF for reading and later text extraction.
pdf_file = "/content/drive/MyDrive/Colab_Notebooks/2021-mazda2_manual.pdf"
pdf = fitz.open(pdf_file)

# Quick check
print(f"Loaded PDF with {pdf.page_count} pages.")

Loaded PDF with 640 pages.


In [21]:
# Extract the entire text from the PDF (text only, images/graphics are ignored)
text = ""
for page in pdf:
    text += page.get_text()

# Quick check
print("Total characters extracted:", len(text))

Total characters extracted: 874282


In [23]:
# -------------------------------
# 5. Basic text cleaning
# -------------------------------
# Clean the extracted text using regex and string replacements:
# - join broken lines
# - fix known ligature / encoding issues from the PDF extraction
# - collapse multiple spaces into one
# - remove non-ASCII characters (keep only standard English text)
import re

# Join broken lines
text = re.sub(r'\n+', ' ', text)

# Fix known PDF extraction issues (specific to this manual)
text = text.replace("À ", "fl").replace("¿ ", "fi").replace("¿", "?").replace("? ", "fi")

# Collapse multiple whitespace characters into a single space
text = re.sub(r"\s+", " ", text) # replace multiple whitespace chars with a single space

# Remove non-ASCII characters
text = re.sub(r"[^\x00-\x7F]+", " ", text) # remove any non-ASCII characters

In [24]:
# -------------------------------
# 6. Chunk text
# -------------------------------
# Split the cleaned text into chunks for embeddings.
# Chunking allows the RAG system to retrieve relevant text sections efficiently.
chunk_size = 1120  # characters per chunk (adjustable)
overlap = 200      # overlapping characters to preserve context across chunks

chunks = []
start = 0
while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start += chunk_size - overlap  # move start for next overlapping chunk

print("Number of chunks:", len(chunks))

Number of chunks: 906


In [25]:
# -------------------------------
# 7. Compute embeddings
# -------------------------------
# Convert chunks to embeddings (dense vector representations) for the text.
# Similar meanings are mapped close together in a semantic space, enabling fast semantic search.

# Using 'all-MiniLM-L6-v2' from sentence-transformers for compact, efficient embeddings.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Compute embeddings for each text chunk
embeddings = model.encode(chunks)

# Quick check
print("Shape of embeddings:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape of embeddings: (906, 384)


In [26]:
# -------------------------------
# 8. Build FAISS index
# -------------------------------
# Store the embeddings in a FAISS vector index for fast similarity search.
# This allows retrieving the most semantically similar text chunks given a query.

dimension = embeddings.shape[1]         # dimensionality of embedding vectors

# Create a flat (non-compressed) L2 index
index = faiss.IndexFlatL2(dimension)    # L2 distance measures similarity

# Add the embeddings to the index
index.add(np.array(embeddings).astype("float32"))  # embeddings should already be float32

# Quick check
print("FAISS index contains", index.ntotal, "vectors")

FAISS index contains 906 vectors


In [37]:
# -------------------------------
# 9. Retrieval Agent (function using Groq)
# -------------------------------
# This function is the first agent in the quasi-agentic RAG workflow.
# It takes a user question and retrieves the top-k (five in this case) most similar text chunks from the FAISS index.

def retrieve_chunks(query, top_k=5):
    """
    Retrieve the top-k most similar text chunks from the FAISS index for a given query.

    Args:
        query (str): The user question.
        top_k (int): Number of top similar chunks to return (default 5).

    Returns:
        list[str]: Retrieved chunks with ranking labels.
    """
    # Convert query to vector
    query_embedding = model.encode([query])

    # Search FAISS index for top_k similar chunks
    distances, indices = index.search(np.array(query_embedding).astype("float32"), k=top_k)

    # Prepare output with ranking labels
    retrieved = []
    for i, idx in enumerate(indices[0]):
        chunk_text = f"Rank {i+1}:\n{chunks[idx].strip()}\n-----"
        retrieved.append(chunk_text)

    return retrieved

In [38]:
# -------------------------------
# 10. LLM Answer Agent (function using Groq)
# -------------------------------
# Two prompts to experiment with the answer style:
# - prompt1: simple and concise
# - prompt2: more informative, faithful to the manual, avoids hallucination

prompt1 = """
You are an assistant that answers questions based on the following car manual text:

{context}

Question: {question}
Answer concisely:
"""

prompt2 = """
You are an assistant that answers questions using ONLY the information from the car manual text below.

Car manual text:
{context}

Question: {question}

Instructions:
- Base your answer only on the provided manual text.
- If the manual contains the exact procedure, explain it clearly.
- If the manual does NOT provide the exact procedure, summarize any relevant information that may help the user (e.g., specifications, warnings, capacities, or related instructions).
- Do not invent information that is not present in the manual.
- Be concise if possible, but include all relevant information.

Answer:
"""

In [39]:
# Second agent in the quasi-agentic RAG workflow.
# Receives chunks from the Retrieval Agent, summarises the content, and returns a presentable answer.
# Ensures the answer is faithful to the manual (prompt2) and avoids hallucination.

#def answer_question(question, top_k=5):
def answer_question(question: str, retrieved: list[str], prompt: str = prompt2) -> str:
    """
    Compose a comprehensive answer to the user's question using retrieved text chunks.

    Args:
        question (str): The user question.
        retrieved (list[str]): Retrieved chunks from the retrieval agent, ranked by similarity.
        prompt (str): Prompt template to use (default is prompt2 for faithful summarization).

    Returns:
        str: Generated answer based on the retrieved chunks.
    """
    # Step 1: Prepare context from retrieved chunks
    context = "\n\n".join(retrieved)   # separate chunks with double newlines for clarity

    # Step 2: Fill the selected prompt template with context and question
    prompt_filled = prompt2.format(context=context, question=question)

    # Step 3: Call Groq LLM to generate answer
    messages = [{"role": "user", "content": prompt_filled}]
    response = client.chat.completions.create(
        model=groq_model,
        messages=messages,
        max_tokens=500, # increased from 150 to allow longer answers
        temperature=0   # deterministic output to reduce hallucination
    )

    # Step 4: Return the final answer as a clean string
    return response.choices[0].message.content.strip()

In [45]:
# -------------------------------
# 11. Orchestrator (User-Facing)
# -------------------------------
# This function coordinates the retrieval and answer agents to provide a response to the user.
# Acts as the leader/coordinator in the quasi-agentic workflow.

def car_manual_assistant(question: str):
    """
    Coordinate the Retrieval Agent and LLM Answer Agent to generate an answer
    to the user's question from the car manual.

    Args:
        question (str): The user question.

    Returns:
        str: The assistant’s answer to the user's question (also printed to console).
    """
    # Step 1: Call the Retrieval agent to retrieve relevant information from the RAG system
    retrieved = retrieve_chunks(question)

    # Step 2: Pass the retrieved chunks to the LLM Answer Agent to have the answer genertaed.
    answer = answer_question(question, retrieved)

    # Step 3: Print the answer.
    print("\nUser Question:\n", question)
    print("\nAssistant Answer:\n", answer)
    print("\n\n")

In [49]:
# -------------------------------
# 12. Test the system
# -------------------------------
q="How do I change the oil in my Mazda 2?"
car_manual_assistant(q)


User Question:
 How do I change the oil in my Mazda 2?

Assistant Answer:
 The manual does not provide the exact procedure for changing the oil in your Mazda 2. However, it recommends that changing the engine oil should be done by an expert repairer, specifically an Authorised Mazda Repairer. 

When replacing the engine oil, it is necessary to inspect the oil level using the oil dipstick and refill so that the engine oil level is within the range between MIN and MAX. The manual also emphasizes the importance of using engine oil with the correct specification to maintain the maintenance interval and protect the engine from damage.

Additionally, after replacing the engine oil, the vehicle's engine control unit needs to be reset, especially for SKYACTIV-D 1.5 models. The manual provides a procedure for resetting the engine control unit on page 6-27, but it is recommended to consult an expert repairer or an Authorised Mazda Repairer for this process.





In [50]:
queries = [
    "What oil should I use?",
    "What is the engine oil capacity for SKYACTIV-D 1.5?",
    "How do I reset the oil warning light after an oil change?"
]

for q in queries:
    car_manual_assistant(q)


User Question:
 What oil should I use?

Assistant Answer:
 To determine the correct oil to use, refer to the recommended SAE viscosity numbers on page 6-23. The manual recommends using Mazda Original Oils, specifically:

- Mazda Original Oil Ultra DPF 5W-30
- Mazda Original Oil Supra DPF 0W-30

Alternative oils meeting the listed specification (ACEA C3, 0W-30, or 5W-30) may also be used. However, it is essential to note that using oils that do not meet the specified requirements may lead to engine damage, which is not covered by the Mazda Warranty.

For African nations, use SL or higher engine oil with the SKYACTIV-G 1.3 and SKYACTIV-G 1.5 LP (Low-Power) engines. For the SKYACTIV-D 1.5 engine, use the specified oil, as using other oil may shorten the Diesel Particulate Filter's effective period or damage it.




User Question:
 What is the engine oil capacity for SKYACTIV-D 1.5?

Assistant Answer:
 The engine oil capacity for SKYACTIV-D 1.5 is as follows: 
- With oil filter replacemen